<a href="https://colab.research.google.com/github/ypg1um-arch/SAU_ML_TASKS/blob/main/Task_4A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
import numpy as np

class DenseLayer:
    """Represents a single fully connected (dense) layer."""
    def __init__(self, input_dim, output_dim, activation='relu'):
        #He (Kaiming) initialization for ReLU, Xavier for Sigmoid
        if activation == 'relu':
            self.weights = np.random.randn(input_dim, output_dim) * np.sqrt(2.0 / input_dim)
        else:
            self.weights = np.random.randn(input_dim, output_dim) * np.sqrt(1.0 / input_dim)

        self.biases = np.zeros((1, output_dim))
        self.activation_type = activation

    def _activate(self, Z):
        """Applies the selected activation function."""
        if self.activation_type == 'relu':
            return np.maximum(0, Z)
        elif self.activation_type == 'sigmoid':
            return 1 / (1 + np.exp(-Z))
        elif self.activation_type == 'softmax':
            # Stabilized softmax to prevent overflow
            exp_Z = np.exp(Z - np.max(Z, axis=-1, keepdims=True))
            return exp_Z / np.sum(exp_Z, axis=-1, keepdims=True)
        else:
            return Z  #Linear/Identity pass-through

    def forward(self, inputs):
        """Executes the forward pass operations for this specific layer."""
        Z = np.dot(inputs, self.weights) + self.biases
        return self._activate(Z)


class MultiLayerPerceptron:
    """Manages the network architecture and executes sequential layer passes."""
    def __init__(self):
        self.layers = []

    def add(self, layer):
        """Appends a DenseLayer to the network."""
        self.layers.append(layer)

    def forward(self, X):
        """Sequentially propagates the input matrix through all network layers."""
        out = X
        for layer in self.layers:
            out = layer.forward(out)
        return out
#Instantiate the MLP architecture
mlp = MultiLayerPerceptron()

#Structure: 4 inputs -> 5 hidden nodes (ReLU) -> 3 hidden nodes (ReLU) -> 2 outputs (Softmax)
mlp.add(DenseLayer(input_dim=4, output_dim=5, activation='relu'))
mlp.add(DenseLayer(input_dim=5, output_dim=3, activation='relu'))
mlp.add(DenseLayer(input_dim=3, output_dim=2, activation='softmax'))

#Define dummy batch matrix: Shape (Batch Size: 2, Features: 4)
X_batch = np.array([
    [0.5, -0.2, 0.1, 1.2],
    [-1.1, 0.4, 0.9, -0.3]
])

#Compute predictions
predictions = mlp.forward(X_batch)

print("Input Batch Shape:", X_batch.shape)
print("Output Batch Shape (Probabilities):", predictions.shape)
print("\nPredicted Class Probabilities:\n", predictions)
print("\nRow validation sums (must equal 1.0 for Softmax):", np.sum(predictions, axis=1))


Input Batch Shape: (2, 4)
Output Batch Shape (Probabilities): (2, 2)

Predicted Class Probabilities:
 [[0.5 0.5]
 [0.5 0.5]]

Row validation sums (must equal 1.0 for Softmax): [1. 1.]


In [50]:
import numpy as np

class DenseLayer:
    def __init__(self, input_dim, output_dim, activation='relu'):
        #He initialization for ReLU, Xavier for Softmax/Identity
        if activation == 'relu':
            self.weights = np.random.randn(input_dim, output_dim) * np.sqrt(2.0 / input_dim)
        else:
            self.weights = np.random.randn(input_dim, output_dim) * np.sqrt(1.0 / input_dim)

        self.biases = np.zeros((1, output_dim))
        self.activation_type = activation

        #Caches for backpropagation
        self.inputs_cache = None
        self.Z_cache = None
        self.A_cache = None

        #Gradients
        self.dW = None
        self.dB = None

    def _activate(self, Z):
        if self.activation_type == 'relu':
            return np.maximum(0, Z)
        elif self.activation_type == 'softmax':
            exp_Z = np.exp(Z - np.max(Z, axis=-1, keepdims=True))
            return exp_Z / np.sum(exp_Z, axis=-1, keepdims=True)
        return Z

    def _activation_derivative(self, dA):
        if self.activation_type == 'relu':
            return dA * (self.Z_cache > 0)
        return dA

    def forward(self, inputs):
        self.inputs_cache = inputs
        self.Z_cache = np.dot(inputs, self.weights) + self.biases
        self.A_cache = self._activate(self.Z_cache)
        return self.A_cache

    def backward(self, dA_or_dZ, is_output=False):
        #Determine dZ (error signal at layer output)
        if is_output:
            dZ = dA_or_dZ
        else:
            dZ = self._activation_derivative(dA_or_dZ)

        #Calculate gradients for parameters
        self.dW = np.dot(self.inputs_cache.T, dZ)
        self.dB = np.sum(dZ, axis=0, keepdims=True)

        #Pass gradient upstream to the previous layer
        dA_prev = np.dot(dZ, self.weights.T)
        return dA_prev


class MultiLayerPerceptron:
    def __init__(self):
        self.layers = []

    def add(self, layer):
        self.layers.append(layer)

    def forward(self, X):
        out = X
        for layer in self.layers:
            out = layer.forward(out)
        return out

    def backward(self, Y_true):
        #Access prediction matrix
        output_layer = self.layers[-1]
        Y_pred = output_layer.A_cache

        #Compute Softmax + Categorical Cross-Entropy derivative
        batch_size = Y_true.shape[0]
        dZ = (Y_pred - Y_true) / batch_size

        #Propagate backward through layers sequentially
        upstream_gradient = output_layer.backward(dZ, is_output=True)
        for layer in reversed(self.layers[:-1]):
            upstream_gradient = layer.backward(upstream_gradient, is_output=False)

    def update_weights(self, learning_rate):
        for layer in self.layers:
            layer.weights -= learning_rate * layer.dW
            layer.biases -= learning_rate * layer.dB


if __name__ == "__main__":
    #Initialize Network (2 Inputs -> 3 Hidden Nodes -> 2 Output Classes)
    mlp = MultiLayerPerceptron()
    mlp.add(DenseLayer(input_dim=2, output_dim=3, activation='relu'))
    mlp.add(DenseLayer(input_dim=3, output_dim=2, activation='softmax'))

    #Mock Data Matrix (Batch Size: 2, Features: 2)
    X_batch = np.array([
        [0.5, -0.8],
        [1.2, 0.3]
    ])

    #One-hot encoded ground truth
    Y_true = np.array([
        [1.0, 0.0],
        [0.0, 1.0]
    ])

    #Forward Pass
    predictions = mlp.forward(X_batch)
    print("--- Forward Pass Predictions ---")
    print(predictions)

    #Backpropagation Pass
    mlp.backward(Y_true)
    print("\nGradients Computed")
    print("Hidden Layer dW Shape:", mlp.layers[0].dW.shape)
    print("Output Layer dW Shape:", mlp.layers[1].dW.shape)

    #Gradient Descent Weight Update
    mlp.update_weights(learning_rate=0.1)
    print("\nStatus: Weights successfully updated without errors.")


--- Forward Pass Predictions ---
[[0.57957068 0.42042932]
 [0.64723808 0.35276192]]

Gradients Computed
Hidden Layer dW Shape: (2, 3)
Output Layer dW Shape: (3, 2)

Status: Weights successfully updated without errors.


In [53]:
import numpy as np

class DenseLayer:
    def __init__(self, input_dim, output_dim, activation='relu'):
        self.activation_type = activation.lower()

        #Specialized Weight Initialization to prevent exploding/vanishing gradients
        if self.activation_type == 'relu':
            self.weights = np.random.randn(input_dim, output_dim) * np.sqrt(2.0 / input_dim)
        elif self.activation_type in ['sigmoid', 'tanh', 'softmax']:
            self.weights = np.random.randn(input_dim, output_dim) * np.sqrt(1.0 / input_dim)
        else:
            self.weights = np.random.randn(input_dim, output_dim) * 0.01

        self.biases = np.zeros((1, output_dim))

        #Caches for backpropagation
        self.inputs_cache = None
        self.Z_cache = None
        self.A_cache = None

        #Gradients
        self.dW = None
        self.dB = None

    def _activate(self, Z):
        """Applies the selected activation function during Forward Pass."""
        if self.activation_type == 'relu':
            return np.maximum(0, Z)

        elif self.activation_type == 'sigmoid':
            return 1.0 / (1.0 + np.exp(-np.clip(Z, -500, 500))) #Clip to prevent overflow

        elif self.activation_type == 'tanh':
            return np.tanh(Z)

        elif self.activation_type == 'softmax':
            exp_Z = np.exp(Z - np.max(Z, axis=-1, keepdims=True))
            return exp_Z / np.sum(exp_Z, axis=-1, keepdims=True)

        return Z #Linear pass-through

    def _activation_derivative(self, dA):
        """Applies the Chain Rule step (dL/dA * dA/dZ) during Backward Pass."""
        if self.activation_type == 'relu':
            return dA * (self.Z_cache > 0)

        elif self.activation_type == 'sigmoid':
            #Derivative: f(z) * (1 - f(z)) -> A * (1 - A)
            return dA * (self.A_cache * (1.0 - self.A_cache))

        elif self.activation_type == 'tanh':
            #Derivative: 1 - tanh^2(z) -> 1 - A^2
            return dA * (1.0 - np.square(self.A_cache))

        return dA

    def forward(self, inputs):
        self.inputs_cache = inputs
        self.Z_cache = np.dot(inputs, self.weights) + self.biases
        self.A_cache = self._activate(self.Z_cache)
        return self.A_cache

    def backward(self, dA_or_dZ, is_output=False):
        if is_output:
            dZ = dA_or_dZ
        else:
            dZ = self._activation_derivative(dA_or_dZ)

        #Calculate weight and bias gradients
        self.dW = np.dot(self.inputs_cache.T, dZ)
        self.dB = np.sum(dZ, axis=0, keepdims=True)

        #Calculate upstream gradient for the previous layer
        dA_prev = np.dot(dZ, self.weights.T)
        return dA_prev


class MultiLayerPerceptron:
    def __init__(self):
        self.layers = []

    def add(self, layer):
        self.layers.append(layer)

    def forward(self, X):
        out = X
        for layer in self.layers:
            out = layer.forward(out)
        return out

    def backward(self, Y_true):
        output_layer = self.layers[-1]
        Y_pred = output_layer.A_cache

        #Compute Softmax + Categorical Cross-Entropy loss gradient directly
        batch_size = Y_true.shape[0]
        dZ = (Y_pred - Y_true) / batch_size

        #Backpropagate through layers sequentially
        upstream_gradient = output_layer.backward(dZ, is_output=True)
        for layer in reversed(self.layers[:-1]):
            upstream_gradient = layer.backward(upstream_gradient, is_output=False)

    def update_weights(self, learning_rate):
        for layer in self.layers:
            layer.weights -= learning_rate * layer.dW
            layer.biases -= learning_rate * layer.dB
if __name__ == "__main__":
    #Create Deep Architecture
    mlp = MultiLayerPerceptron()
    mlp.add(DenseLayer(input_dim=5, output_dim=4, activation='relu'))
    mlp.add(DenseLayer(input_dim=4, output_dim=3, activation='sigmoid'))
    mlp.add(DenseLayer(input_dim=3, output_dim=3, activation='tanh'))
    mlp.add(DenseLayer(input_dim=3, output_dim=2, activation='softmax'))

    #Mock Data: 3 items, 5 features each
    X_batch = np.random.randn(3, 5)
    Y_true = np.array([
        [1.0, 0.0],
        [0.0, 1.0],
        [1.0, 0.0]
    ])

    #Run full system loop
    preds = mlp.forward(X_batch)
    print("Forward Pass Out (Softmax):\n", preds)

    mlp.backward(Y_true)
    mlp.update_weights(learning_rate=0.05)
    print("\nStatus: Forward and Backpropagation completed smoothly across all activations.")


Forward Pass Out (Softmax):
 [[0.5831619  0.4168381 ]
 [0.5560294  0.4439706 ]
 [0.63074166 0.36925834]]

Status: Forward and Backpropagation completed smoothly across all activations.


In [60]:
import numpy as np

class MultiLayerPerceptron:
    def __init__(self):
        self.layers = []

    def add(self, layer):
        self.layers.append(layer)

    def forward(self, X):
        out = X
        for layer in self.layers:
            out = layer.forward(out)
        return out

    def backward(self, Y_true):
        output_layer = self.layers[-1]
        Y_pred = output_layer.A_cache

        #Softmax + Categorical Cross-Entropy loss gradient
        batch_size = Y_true.shape[0]
        dZ = (Y_pred - Y_true) / batch_size

        upstream_gradient = output_layer.backward(dZ, is_output=True)
        for layer in reversed(self.layers[:-1]):
            upstream_gradient = layer.backward(upstream_gradient, is_output=False)

    def update_weights(self, learning_rate):
        for layer in self.layers:
            layer.weights -= learning_rate * layer.dW
            layer.biases -= learning_rate * layer.dB

    def compute_loss(self, Y_pred, Y_true):
        """Computes Categorical Cross-Entropy Loss."""
        batch_size = Y_true.shape[0]
        #Clip to prevent log(0) undefined errors
        Y_pred = np.clip(Y_pred, 1e-15, 1.0 - 1e-15)
        loss = -np.sum(Y_true * np.log(Y_pred)) / batch_size
        return loss

    def fit(self, X, Y, epochs, batch_size, learning_rate):
        """Trains the network using Mini-Batch Gradient Descent."""
        num_samples = X.shape[0]

        for epoch in range(epochs):
            #Shuffle dataset at the start of every epoch
            indices = np.arange(num_samples)
            np.random.shuffle(indices)
            X_shuffled = X[indices]
            Y_shuffled = Y[indices]

            epoch_loss = 0.0
            num_batches = int(np.ceil(num_samples / batch_size))

            #Iterate through mini-batches
            for b in range(num_batches):
                start_idx = b * batch_size
                end_idx = min(start_idx + batch_size, num_samples)

                X_batch = X_shuffled[start_idx:end_idx]
                Y_batch = Y_shuffled[start_idx:end_idx]

                #Core optimization step per batch
                predictions = self.forward(X_batch)
                batch_loss = self.compute_loss(predictions, Y_batch)
                epoch_loss += batch_loss * (end_idx - start_idx) #weighted loss for uneven batches

                self.backward(Y_batch)
                self.update_weights(learning_rate)

            #Print status update
            epoch_loss /= num_samples
            if (epoch + 1) % max(1, epochs // 10) == 0 or epoch == 0:
                print(f"Epoch {epoch+1:03d}/{epochs:03d} -> Loss: {epoch_loss:.4f}")
if __name__ == "__main__":
    #Generate Fake Dataset (100 samples, 4 features)
    np.random.seed(42)
    X_train = np.random.randn(100, 4)

    #Generate 3-class One-Hot Encoded Targets
    random_classes = np.random.randint(0, 3, size=100)
    Y_train = np.zeros((100, 3))
    Y_train[np.arange(100), random_classes] = 1.0

    #Configure Deep MLP Network
    mlp = MultiLayerPerceptron()
    mlp.add(DenseLayer(input_dim=4, output_dim=8, activation='relu'))
    mlp.add(DenseLayer(input_dim=8, output_dim=6, activation='tanh'))
    mlp.add(DenseLayer(input_dim=6, output_dim=3, activation='softmax'))

    #Run Mini-Batch Gradient Descent Training Loop
    print("Starting Mini-Batch Training Loop:")
    mlp.fit(
        X=X_train,
        Y=Y_train,
        epochs=100,
        batch_size=16,    #Process 16 samples per optimization step
        learning_rate=0.1
    )


Starting Mini-Batch Training Loop:
Epoch 001/100 -> Loss: 1.1690
Epoch 010/100 -> Loss: 1.0282
Epoch 020/100 -> Loss: 0.9734
Epoch 030/100 -> Loss: 0.9557
Epoch 040/100 -> Loss: 0.9266
Epoch 050/100 -> Loss: 0.9095
Epoch 060/100 -> Loss: 0.8864
Epoch 070/100 -> Loss: 0.8543
Epoch 080/100 -> Loss: 0.8406
Epoch 090/100 -> Loss: 0.8179
Epoch 100/100 -> Loss: 0.7904


In [66]:
import numpy as np

#Use the DenseLayer and MultiLayerPerceptron classes defined previously

def gradient_check(mlp, X, Y, epsilon=1e-5):
    """
    Compares analytical gradients (finite differences) with backpropagation gradients.
    Returns the relative difference.
    """
    #Run standard forward and backward pass to get backprop gradients
    preds = mlp.forward(X)
    mlp.backward(Y)

    #We will check the weights of the first layer as an example
    target_layer = mlp.layers[0]
    backprop_grads = np.copy(target_layer.dW)
    numerical_grads = np.zeros_like(target_layer.weights)

    #Compute Analytical (Numerical) Gradients element by element
    #Flatten weights to make looping trivial
    it = np.nditer(target_layer.weights, flags=['multi_index'], op_flags=['readwrite'])
    while not it.finished:
        idx = it.multi_index
        original_value = target_layer.weights[idx]

        #Calculate L(W + epsilon)
        target_layer.weights[idx] = original_value + epsilon
        loss_plus = mlp.compute_loss(mlp.forward(X), Y)

        # Calculate L(W - epsilon)
        target_layer.weights[idx] = original_value - epsilon
        loss_minus = mlp.compute_loss(mlp.forward(X), Y)

        #Reset weight to its original value
        target_layer.weights[idx] = original_value

        #Central difference formula
        numerical_grads[idx] = (loss_plus - loss_minus) / (2 * epsilon)
        it.iternext()

    #Calculate Relative Difference
    numerator = np.linalg.norm(backprop_grads - numerical_grads)
    denominator = np.linalg.norm(backprop_grads) + np.linalg.norm(numerical_grads)
    relative_difference = numerator / denominator

    return relative_difference, backprop_grads, numerical_grads

if __name__ == "__main__":
    #Create a small network to make numerical looping fast
    mlp = MultiLayerPerceptron()
    mlp.add(DenseLayer(input_dim=3, output_dim=4, activation='relu'))
    mlp.add(DenseLayer(input_dim=4, output_dim=2, activation='softmax'))

    #Tiny dataset (1 sample, 3 features, 2 classes)
    X_test = np.array([[0.5, -0.2, 0.1]])
    Y_test = np.array([[1.0, 0.0]])

    #Run comparison
    diff, bp_g, num_g = gradient_check(mlp, X_test, Y_test)

    print("Gradient Check Results:")
    print(f"Relative Difference: {diff:.4e}")

    if diff < 1e-7:
        print("Backpropagation is mathematically correct! (Diff < 1e-7)")
    else:
        print("Gradient mismatch detected. Review backpropagation chain rule.")

    print("\nSample Backprop Gradients (First 2x2 block):\n", bp_g[:2, :2])
    print("Sample Analytical Gradients (First 2x2 block):\n", num_g[:2, :2])


Gradient Check Results:
Relative Difference: 1.6691e-11
Backpropagation is mathematically correct! (Diff < 1e-7)

Sample Backprop Gradients (First 2x2 block):
 [[ 0.15435038  0.        ]
 [-0.06174015  0.        ]]
Sample Analytical Gradients (First 2x2 block):
 [[ 0.15435038  0.        ]
 [-0.06174015  0.        ]]
